# BioGen 2024 — Phase 1 Base Notebook

It covers:
- loading the BioGen topics and PubMed abstracts
- building OpenSearch indexes
- running **BM25**, **LM Jelinek-Mercer**, **LM Dirichlet**, and **dense k-NN** retrieval
- producing ranked runs
- evaluating with **Precision@10**, **Recall@100**, **NDCG**, and a simple **precision-recall curve**
- splitting topics into **train (odd ids)** and **test (even ids)**


In [1]:
# Uncomment if needed
# !pip install opensearch-py sentence-transformers scikit-learn matplotlib pandas tqdm

In [2]:
from __future__ import annotations

import json
import math
import os
from pathlib import Path
from collections import defaultdict
import pprint as pp

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm.auto import tqdm
from opensearchpy import OpenSearch, helpers
from opensearchpy.helpers import bulk

# Optional, only needed for dense retrieval
USE_DENSE = True
if USE_DENSE:
    from sentence_transformers import SentenceTransformer

## 1) Configuration

Set your local paths and your OpenSearch connection.

In [3]:
# ---------------------------
# File paths
# ---------------------------
DATA_DIR = Path("./BioGen2024")

TOPICS_PATH = DATA_DIR / "BioGen2024topics.json"
ABSTRACTS_PATH = DATA_DIR / "filtered_pubmed_abstracts.txt"
SUBMISSIONS_PATH = DATA_DIR / "biogen_2024_submissions.json"

# Ground truth for Phase 1 evaluation
# Expected format example:
# {
#   "116": {"33659106": 1, "14971838": 1, "24118690": 1},
#   "117": {"12345678": 1}
# }
QRELS_PATH = DATA_DIR / "biogen_2024_qrels.json"  # create/update when you have the official ground truth

# ---------------------------
# OpenSearch connection
# ---------------------------
OS_HOST = "api.novasearch.org"
OS_PORT = 443
OS_USER = "usernlp16"
OS_PASS = "hEatz?gz5K"

index_name = OS_USER

# ---------------------------
# Index names
# ---------------------------
BM25_INDEX = "biogen_phase1_bm25"
LMJM_INDEX = "biogen_phase1_lmjm"
LMDIR_INDEX = "biogen_phase1_lmdir"
KNN_INDEX = "biogen_phase1_knn"

# ---------------------------
# Dense retrieval model
# ---------------------------
EMBEDDING_MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
# Replace later with a biomedical encoder if you want.

# ---------------------------
# Experiment settings
# ---------------------------
TOP_K_BM25 = 100
TOP_K_KNN = 100
BATCH_SIZE = 512
QUERY_MODE = "all"   # one of: "topic", "question", "narrative", "all"

## 2) Load the data

In [ ]:
def load_topics(path: Path) -> pd.DataFrame:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    topics = data["topics"]
    df = pd.DataFrame(topics)
    df["id"] = df["id"].astype(int)
    return df

def load_jsonl_abstracts(path: Path) -> pd.DataFrame:
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            rows.append(json.loads(line))
    df = pd.DataFrame(rows)
    df["id"] = df["id"].astype(str)
    return df

def load_optional_json(path: Path):
    if path.exists():
        with open(path, "r", encoding="utf-8") as f:
            return json.load(f)
    return None

topics_df = load_topics(TOPICS_PATH)
abstracts_df = load_jsonl_abstracts(ABSTRACTS_PATH)
submissions = load_optional_json(SUBMISSIONS_PATH)

print("Topics:", topics_df.shape)
print("Abstracts:", abstracts_df.shape)
print("Submissions loaded:", submissions is not None)

topics_df.head()

Topics: (65, 4)
Abstracts: (4194, 2)
Submissions loaded: True


,id,contents
0,61500,Oral methionine in the treatment of severe par...
1,89388,Cold pressor test in detection of coronary hea...
2,155033,Origin of the extra chromosome in trisomy 21.\...
3,347160,[Treatment of herpes simplex keratitis with Vi...
4,375859,"High blood pressure. A side effect of drugs, p..."


In [15]:
abstracts_df.head()

,id,contents
0,61500,Oral methionine in the treatment of severe par...
1,89388,Cold pressor test in detection of coronary hea...
2,155033,Origin of the extra chromosome in trisomy 21.\...
3,347160,[Treatment of herpes simplex keratitis with Vi...
4,375859,"High blood pressure. A side effect of drugs, p..."


In [5]:
def build_query_text(row: pd.Series, mode: str = "all") -> str:
    mode = mode.lower()
    if mode == "topic":
        return str(row["topic"]).strip()
    if mode == "question":
        return str(row["question"]).strip()
    if mode == "narrative":
        return str(row["narrative"]).strip()
    if mode == "all":
        return " ".join([
            str(row["topic"]).strip(),
            str(row["question"]).strip(),
            str(row["narrative"]).strip(),
        ]).strip()
    raise ValueError(f"Unknown query mode: {mode}")

topics_df["query_text"] = topics_df.apply(build_query_text, axis=1, mode=QUERY_MODE)
topics_df[["id", "topic", "query_text"]].head(3)

,id,topic,query_text
0,116,natural treatments for sleep apnea,natural treatments for sleep apnea Are there w...
1,117,runx2 mutations,runx2 mutations What will mutation in runx2 af...
2,118,cornea injury healing,cornea injury healing How long does a minor co...


In [6]:
def split_train_test_by_topic_id(df: pd.DataFrame):
    train_df = df[df["id"] % 2 == 1].copy()  # odd
    test_df = df[df["id"] % 2 == 0].copy()   # even
    return train_df, test_df

train_topics_df, test_topics_df = split_train_test_by_topic_id(topics_df)

print("Train topics (odd ids):", len(train_topics_df))
print("Test topics (even ids):", len(test_topics_df))

Train topics (odd ids): 32
Test topics (even ids): 33


## 3) Connect to OpenSearch

In [9]:
client = OpenSearch(
    hosts = [{'host': OS_HOST, 'port': OS_PORT}],
    http_compress = True, # enables gzip compression for request bodies
    http_auth = (OS_USER, OS_PASS),
    use_ssl = True,
    url_prefix = 'opensearch_v3',
    verify_certs = False,
    ssl_assert_hostname = False,
    ssl_show_warn = False
)

if client.indices.exists(index=index_name):

    resp = client.indices.open(index=index_name)
    print(resp)

    print('\n----------------------------------------------------------------------------------- INDEX SETTINGS')
    settings = client.indices.get_settings(index=index_name)
    pp.pprint(settings)

    print('\n----------------------------------------------------------------------------------- INDEX MAPPINGS')
    mappings = client.indices.get_mapping(index=index_name)
    pp.pprint(mappings)

    print('\n----------------------------------------------------------------------------------- INDEX #DOCs')
    print(client.count(index=index_name))
else:
    print("Index does not exist.")

Index does not exist.


## 4) Create indexes

For lexical retrieval, it is easiest to maintain **separate indexes** so each field can use a different similarity:
- BM25
- LM Jelinek-Mercer
- LM Dirichlet

For dense retrieval, we create a k-NN vector index.

In [14]:
# if client.indices.exists(index=index_name):
#     client.indices.delete(index=index_name)
#     print(f"Deleted index {index_name}")

client.indices.delete(index=index_name)

index_body = {
   "settings": {
      "index": {
         "number_of_replicas": 0,
         "number_of_shards": 4,
         "refresh_interval": "-1",
         "knn": True
      }
   },
   "mappings": {
      "dynamic": "strict",
      "properties": {
         "doc_id": {
            "type": "keyword"
         },
         "bm25_content": {
            "type": "text",
            "analyzer": "standard",
            "similarity": "BM25"
         },
         "jelineck_content": {
            "type": "text",
            "analyzer": "standard",
            "similarity": "LMJelinekMercer"
         },
         "dirichlet_content": {
            "type": "text",
            "analyzer": "standard",
            "similarity": "LMDirichlet"
         }
      }
   }
}

if client.indices.exists(index=index_name):
    print("Index already existed. Nothing to be done.")
else:        
    response = client.indices.create(index=index_name, body=index_body)
    print('\nCreating index:')
    print(response)

RequestError: RequestError(400, 'mapper_parsing_exception', 'Unknown Similarity type [LMDirichlet] for field [dirichlet_content]')

In [13]:
actions = []

with open(ABSTRACTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        doc = json.loads(line)

        doc_id = doc["id"]
        contents = doc["contents"]

        action = {
            "_index": index_name,
            "_id": doc_id,
            "_source": {
                "doc_id": doc_id,
                "bm25_content": contents,
                "jelineck_content": contents,
                "dirichlet_content": contents
            }
        }

        actions.append(action)

success, errors = bulk(client, actions)
client.indices.refresh(index=index_name)
print(f"Indexed {success} documents. Errors: {errors}")

BulkIndexError: ('500 document(s) failed to index.', [{'index': {'_index': 'usernlp16', '_id': '61500', 'status': 400, 'error': {'type': 'strict_dynamic_mapping_exception', 'reason': 'mapping set to strict, dynamic introduction of [jelineck_content] within [_doc] is not allowed'}, 'data': {'doc_id': '61500', 'bm25_content': 'Oral methionine in the treatment of severe paracetamol (Acetaminophen) overdose.\n30 patients at risk of hepatic damage from paracetamol (acetaminophen) ingestion were given 2-5 g oral methionine every four hours up to a total dose of 10 g. The first dose was given within ten hours of the overdose. There were no deaths and no reports of hepatic encephalopathy or other complications. In 21 patients plasma aspartate-aminotransferase remained within normal limits. These results suggest that methionine may be effective in reducing the frequency and severity of paracetamol-induced liver damage and may provide an effective non-toxic alternative to cysteamine.\n', 'jeline

In [ ]:
# Create the three lexical indexes

# Create dense index if enabled
# model = None
# if USE_DENSE:
#     model = SentenceTransformer(EMBEDDING_MODEL_NAME)
#     vector_dim = int(model.get_sentence_embedding_dimension())
#     delete_index_if_exists(KNN_INDEX)
#     create_knn_index(KNN_INDEX, vector_dim=vector_dim)

AuthorizationException: AuthorizationException(403, '')

## 5) Index the PubMed abstracts

In [ ]:
def make_text_actions(df: pd.DataFrame, index_name: str):
    for row in df.itertuples(index=False):
        yield {
            "_index": index_name,
            "_id": str(row.id),
            "_source": {
                "pmid": str(row.id),
                "contents": row.contents
            }
        }

def make_knn_actions(df: pd.DataFrame, index_name: str, encoder):
    texts = df["contents"].tolist()
    pmids = df["id"].astype(str).tolist()
    vectors = encoder.encode(texts, batch_size=64, show_progress_bar=True, normalize_embeddings=True)
    for pmid, text, vector in zip(pmids, texts, vectors):
        yield {
            "_index": index_name,
            "_id": pmid,
            "_source": {
                "pmid": pmid,
                "contents": text,
                "vector": vector.tolist()
            }
        }

# Lexical indexes
for idx in [BM25_INDEX, LMJM_INDEX, LMDIR_INDEX]:
    helpers.bulk(client, make_text_actions(abstracts_df, idx), chunk_size=BATCH_SIZE, request_timeout=300)
    client.indices.refresh(index=idx)
    print(f"Indexed documents into {idx}")

# Dense index
if USE_DENSE:
    helpers.bulk(client, make_knn_actions(abstracts_df, KNN_INDEX, model), chunk_size=128, request_timeout=300)
    client.indices.refresh(index=KNN_INDEX)
    print(f"Indexed documents into {KNN_INDEX}")

## 6) Retrieval functions

In [ ]:
def lexical_search(index_name: str, query_text: str, top_k: int = 100):
    body = {
        "size": top_k,
        "query": {
            "match": {
                "contents": {
                    "query": query_text
                }
            }
        }
    }
    resp = client.search(index=index_name, body=body)
    hits = resp["hits"]["hits"]
    return [(hit["_source"]["pmid"], float(hit["_score"])) for hit in hits]

def knn_search(index_name: str, query_text: str, encoder, top_k: int = 100):
    query_vector = encoder.encode([query_text], normalize_embeddings=True)[0].tolist()
    body = {
        "size": top_k,
        "query": {
            "knn": {
                "vector": {
                    "vector": query_vector,
                    "k": top_k
                }
            }
        }
    }
    resp = client.search(index=index_name, body=body)
    hits = resp["hits"]["hits"]
    return [(hit["_source"]["pmid"], float(hit["_score"])) for hit in hits]

In [ ]:
# Quick sanity check with topic 116
example_row = topics_df.iloc[0]
example_query = example_row["query_text"]

bm25_preview = lexical_search(BM25_INDEX, example_query, top_k=5)
jm_preview = lexical_search(LMJM_INDEX, example_query, top_k=5)
dir_preview = lexical_search(LMDIR_INDEX, example_query, top_k=5)

print("Query:", example_query)
print("BM25 preview:", bm25_preview[:3])
print("LMJM preview:", jm_preview[:3])
print("LMDIR preview:", dir_preview[:3])

if USE_DENSE:
    knn_preview = knn_search(KNN_INDEX, example_query, model, top_k=5)
    print("KNN preview:", knn_preview[:3])

## 7) Run retrieval over all topics

In [ ]:
def build_run(topics: pd.DataFrame, method_name: str) -> dict[str, dict[str, float]]:
    run = {}
    for row in tqdm(topics.itertuples(index=False), total=len(topics), desc=method_name):
        qid = str(row.id)
        query_text = row.query_text

        if method_name == "bm25":
            results = lexical_search(BM25_INDEX, query_text, top_k=TOP_K_BM25)
        elif method_name == "lmjm":
            results = lexical_search(LMJM_INDEX, query_text, top_k=TOP_K_BM25)
        elif method_name == "lmdir":
            results = lexical_search(LMDIR_INDEX, query_text, top_k=TOP_K_BM25)
        elif method_name == "knn":
            if not USE_DENSE:
                raise RuntimeError("USE_DENSE=False but method_name='knn'")
            results = knn_search(KNN_INDEX, query_text, model, top_k=TOP_K_KNN)
        else:
            raise ValueError(f"Unknown method: {method_name}")

        run[qid] = {doc_id: score for doc_id, score in results}
    return run

bm25_run = build_run(topics_df, "bm25")
lmjm_run = build_run(topics_df, "lmjm")
lmdir_run = build_run(topics_df, "lmdir")
knn_run = build_run(topics_df, "knn") if USE_DENSE else None

print("Built runs.")

In [ ]:
def save_run(run: dict[str, dict[str, float]], path: Path):
    path.parent.mkdir(parents=True, exist_ok=True)
    with open(path, "w", encoding="utf-8") as f:
        json.dump(run, f, ensure_ascii=False, indent=2)

RUNS_DIR = Path("./runs")
save_run(bm25_run, RUNS_DIR / "bm25_run.json")
save_run(lmjm_run, RUNS_DIR / "lmjm_run.json")
save_run(lmdir_run, RUNS_DIR / "lmdir_run.json")
if knn_run is not None:
    save_run(knn_run, RUNS_DIR / "knn_run.json")

print("Saved runs to:", RUNS_DIR.resolve())

## 8) Evaluation

The official Phase 1 evaluation requires:
- Precision@10
- Recall@100
- NDCG
- precision-recall curves

This section expects the official **qrels / ground truth** file.

### Expected qrels format
```json
{
  "116": {"33659106": 1, "14971838": 1, "24118690": 1},
  "117": {"12345678": 1}
}
```

You can use graded relevance too:
```json
{
  "116": {"33659106": 2, "14971838": 1}
}
```

In [ ]:
def load_qrels(path: Path) -> dict[str, dict[str, int]]:
    if not path.exists():
        raise FileNotFoundError(
            f"Qrels file not found: {path}\n"
            "Download the official ground truth and save it at QRELS_PATH."
        )
    with open(path, "r", encoding="utf-8") as f:
        qrels = json.load(f)
    return {str(k): {str(doc): int(rel) for doc, rel in v.items()} for k, v in qrels.items()}

In [ ]:
def ranked_doc_ids(run_for_qid: dict[str, float], k: int | None = None) -> list[str]:
    ranked = sorted(run_for_qid.items(), key=lambda x: x[1], reverse=True)
    docs = [doc_id for doc_id, _ in ranked]
    return docs if k is None else docs[:k]

def precision_at_k(qrels_for_qid: dict[str, int], run_for_qid: dict[str, float], k: int = 10) -> float:
    ranked = ranked_doc_ids(run_for_qid, k)
    if not ranked:
        return 0.0
    rel_count = sum(1 for doc_id in ranked if qrels_for_qid.get(doc_id, 0) > 0)
    return rel_count / k

def recall_at_k(qrels_for_qid: dict[str, int], run_for_qid: dict[str, float], k: int = 100) -> float:
    total_rel = sum(1 for _, rel in qrels_for_qid.items() if rel > 0)
    if total_rel == 0:
        return 0.0
    ranked = ranked_doc_ids(run_for_qid, k)
    rel_count = sum(1 for doc_id in ranked if qrels_for_qid.get(doc_id, 0) > 0)
    return rel_count / total_rel

def dcg_at_k(qrels_for_qid: dict[str, int], run_for_qid: dict[str, float], k: int = 10) -> float:
    ranked = ranked_doc_ids(run_for_qid, k)
    dcg = 0.0
    for i, doc_id in enumerate(ranked, start=1):
        rel = qrels_for_qid.get(doc_id, 0)
        if rel > 0:
            dcg += (2**rel - 1) / math.log2(i + 1)
    return dcg

def idcg_at_k(qrels_for_qid: dict[str, int], k: int = 10) -> float:
    ideal_rels = sorted([rel for rel in qrels_for_qid.values() if rel > 0], reverse=True)[:k]
    idcg = 0.0
    for i, rel in enumerate(ideal_rels, start=1):
        idcg += (2**rel - 1) / math.log2(i + 1)
    return idcg

def ndcg_at_k(qrels_for_qid: dict[str, int], run_for_qid: dict[str, float], k: int = 10) -> float:
    idcg = idcg_at_k(qrels_for_qid, k=k)
    if idcg == 0:
        return 0.0
    return dcg_at_k(qrels_for_qid, run_for_qid, k=k) / idcg

def evaluate_run(qrels: dict[str, dict[str, int]], run: dict[str, dict[str, float]]) -> dict[str, float]:
    p10, r100, ndcg10 = [], [], []
    for qid, qrels_for_qid in qrels.items():
        run_for_qid = run.get(qid, {})
        p10.append(precision_at_k(qrels_for_qid, run_for_qid, k=10))
        r100.append(recall_at_k(qrels_for_qid, run_for_qid, k=100))
        ndcg10.append(ndcg_at_k(qrels_for_qid, run_for_qid, k=10))
    return {
        "P@10": float(np.mean(p10)) if p10 else 0.0,
        "Recall@100": float(np.mean(r100)) if r100 else 0.0,
        "NDCG@10": float(np.mean(ndcg10)) if ndcg10 else 0.0,
    }

def precision_recall_curve_for_run(qrels: dict[str, dict[str, int]], run: dict[str, dict[str, float]], max_k: int = 100):
    recalls = []
    precisions = []
    for k in range(1, max_k + 1):
        pk_all = []
        rk_all = []
        for qid, qrels_for_qid in qrels.items():
            run_for_qid = run.get(qid, {})
            ranked = ranked_doc_ids(run_for_qid, k)
            rel_total = sum(1 for rel in qrels_for_qid.values() if rel > 0)
            if rel_total == 0:
                continue
            rel_found = sum(1 for doc_id in ranked if qrels_for_qid.get(doc_id, 0) > 0)
            pk = rel_found / k
            rk = rel_found / rel_total
            pk_all.append(pk)
            rk_all.append(rk)
        precisions.append(float(np.mean(pk_all)) if pk_all else 0.0)
        recalls.append(float(np.mean(rk_all)) if rk_all else 0.0)
    return recalls, precisions

In [ ]:
# Run this cell only after you have the official qrels file.
if QRELS_PATH.exists():
    qrels = load_qrels(QRELS_PATH)

    results = []
    for name, run in {
        "BM25": bm25_run,
        "LM-JelinekMercer": lmjm_run,
        "LM-Dirichlet": lmdir_run,
        **({"KNN": knn_run} if knn_run is not None else {})
    }.items():
        metrics = evaluate_run(qrels, run)
        metrics["method"] = name
        results.append(metrics)

    results_df = pd.DataFrame(results)[["method", "P@10", "Recall@100", "NDCG@10"]]
    display(results_df.sort_values("NDCG@10", ascending=False))
else:
    print("Qrels file not found yet. Add the official ground truth to evaluate the runs.")

In [ ]:
# Precision-Recall curves
if QRELS_PATH.exists():
    qrels = load_qrels(QRELS_PATH)

    plt.figure(figsize=(7, 5))
    for name, run in {
        "BM25": bm25_run,
        "LM-JelinekMercer": lmjm_run,
        "LM-Dirichlet": lmdir_run,
        **({"KNN": knn_run} if knn_run is not None else {})
    }.items():
        recalls, precisions = precision_recall_curve_for_run(qrels, run, max_k=100)
        plt.plot(recalls, precisions, label=name)

    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title("Precision-Recall Curve")
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.show()
else:
    print("Qrels file not found yet. Add the official ground truth to plot PR curves.")

## 9) Compare train vs test topics

Use the official split:
- odd topic ids = train
- even topic ids = test

In [ ]:
def subset_qrels(qrels: dict[str, dict[str, int]], topic_ids: list[int]) -> dict[str, dict[str, int]]:
    keep = {str(tid) for tid in topic_ids}
    return {qid: docs for qid, docs in qrels.items() if qid in keep}

if QRELS_PATH.exists():
    qrels = load_qrels(QRELS_PATH)

    qrels_train = subset_qrels(qrels, train_topics_df["id"].tolist())
    qrels_test = subset_qrels(qrels, test_topics_df["id"].tolist())

    rows = []
    runs_to_eval = {
        "BM25": bm25_run,
        "LM-JelinekMercer": lmjm_run,
        "LM-Dirichlet": lmdir_run,
        **({"KNN": knn_run} if knn_run is not None else {})
    }

    for split_name, split_qrels in [("train", qrels_train), ("test", qrels_test)]:
        for method_name, run in runs_to_eval.items():
            metrics = evaluate_run(split_qrels, run)
            rows.append({
                "split": split_name,
                "method": method_name,
                **metrics
            })

    split_results_df = pd.DataFrame(rows)
    display(split_results_df.sort_values(["split", "NDCG@10"], ascending=[True, False]))
else:
    print("Qrels file not found yet. Add the official ground truth to compare train/test.")

## 10) Error analysis helper

This helps inspect which documents were retrieved for a specific query.

In [ ]:
def inspect_results(qid: int | str, run: dict[str, dict[str, float]], abstracts: pd.DataFrame, top_k: int = 5):
    qid = str(qid)
    ranked = sorted(run.get(qid, {}).items(), key=lambda x: x[1], reverse=True)[:top_k]
    rows = []
    for rank, (doc_id, score) in enumerate(ranked, start=1):
        match = abstracts[abstracts["id"] == str(doc_id)]
        text = match.iloc[0]["contents"] if len(match) else ""
        rows.append({
            "rank": rank,
            "pmid": doc_id,
            "score": score,
            "contents_preview": text[:500]
        })
    return pd.DataFrame(rows)

inspect_results(116, bm25_run, abstracts_df, top_k=5)

## 11) Notes for your report

Good ablations for Phase 1:
- query field choice: topic vs question vs narrative vs all
- preprocessing variants
- BM25 parameter tuning (`k1`, `b`)
- LM Jelinek-Mercer lambda tuning
- LM Dirichlet mu tuning
- dense model choice
- hybrid retrieval (later, if you want an extension)

Good tables/figures:
- overall metrics table
- train vs test table
- precision-recall curve
- example successes/failures for 2–3 topics

## 12) Next step after Phase 1

Once you are happy with the retrieval results:
1. keep the best retrieval method or top-N candidate set
2. move to Phase 2 sentence selection
3. use those retrieved abstracts as the source for grounded generation